# Main Course — Reinforcement Learning with Stable Baselines 3

Introductory walkthrough: load a Gymnasium environment, train a PPO agent, evaluate it, and experiment with callbacks, custom policies, and alternate algorithms.

**Reference links**
- [Stable Baselines 3 — RL Guide](https://stable-baselines3.readthedocs.io/en/master/guide/rl.html)
- [Spinning Up — RL Algorithms Taxonomy](https://spinningup.openai.com/en/latest/spinningup/rl_intro2.html#a-taxonomy-of-rl-algorithms)

# 1. Import Gymnasium (first)

Import Gymnasium **before** Stable Baselines 3. SB3 pulls in OpenCV, which bundles SDL2 — the same library pygame uses for `env.render()`. On macOS, loading both causes SDL conflicts if you render after importing SB3.

In [ ]:
# Uncomment to install dependencies inside the notebook (prefer `make setup` for this project)
# !pip install stable-baselines3[extra]

In [35]:
# Paths for best-model checkpoint and TensorBoard logs
import os

# Use os.path.join for each segment — str.join() on a path joins CHARACTERS, not folders
training_path = os.path.join("Training")
saved_path = os.path.join(training_path, "Saved Models")
logs_path = os.path.join(training_path, "Logs")
print(f"Training: {training_path}, Models: {saved_path}, Logs: {logs_path}")

Training: Training, Models: Training/Saved Models, Logs: Training/Logs


In [36]:
import gymnasium as gym  # Environment API (imported as gym for tutorial compatibility)

# 2. Explore the Environment (before SB3)

Run the next cells **before** importing Stable Baselines 3 so live rendering works without SDL warnings on macOS.

In [37]:
# CartPole-v1: balance a pole on a cart by pushing left (0) or right (1)
env_name = "CartPole-v1"

In [38]:
# render_mode="human" opens a live pygame window for env.render()
# Must run before SB3 import — OpenCV (loaded by SB3) also bundles SDL2 on macOS
env = gym.make(env_name, render_mode="human")

In [39]:
# Random baseline episodes with live rendering (run BEFORE importing SB3 below)
episodes = 5
for episode in range(episodes):
    state, _ = env.reset()  # Gymnasium reset returns (observation, info)
    done = False
    score = 0

    while not done:
        env.render()  # Triggers pygame SDL — safe only before OpenCV/SB3 is imported
        action = env.action_space.sample()  # 0 = push left, 1 = push right
        n_state, reward, terminated, truncated, info = env.step(action)
        done = terminated or truncated
        score += reward

    print(f"Episode: {episode + 1}, Score: {score}")

env.close()

Episode: 1, Score: 14.0
Episode: 2, Score: 13.0
Episode: 3, Score: 24.0
Episode: 4, Score: 14.0
Episode: 5, Score: 24.0


### Import Stable Baselines 3

From here on, avoid `render_mode="human"` and `env.render()` on macOS — use `render=False` in evaluation instead.

In [40]:
from stable_baselines3 import PPO  # Proximal Policy Optimization
from stable_baselines3.common.vec_env import DummyVecEnv  # Vectorized env wrapper for SB3
from stable_baselines3.common.monitor import Monitor  # Logs episode reward/length for eval
from stable_baselines3.common.evaluation import evaluate_policy


def make_env(render_mode=None):
    """Create a Monitor-wrapped CartPole env for SB3 training and evaluation."""
    env = gym.make(env_name, render_mode=render_mode)
    # Monitor logs true episode reward/length — SB3 expects this for accurate evaluation
    env = Monitor(env)
    return env

## Understanding the Environment

CartPole observation and action spaces define what the agent sees and can do.

Source: [CartPole environment implementation](https://github.com/Farama-Foundation/Gymnasium/blob/main/gymnasium/envs/classic_control/cartpole.py)

In [41]:
# Action space: Discrete(2) — 0 = push cart left, 1 = push cart right
env = make_env()
env.action_space.sample()

np.int64(0)

In [42]:
# Observation space: Box(4,) — [cart position, cart velocity, pole angle, pole angular velocity]
env.observation_space.sample()

array([ 2.145474  ,  0.5726293 ,  0.04387063, -0.9252096 ], dtype=float32)

# 3. Train an RL Model

Wrap the environment for SB3 and train a PPO agent with TensorBoard logging.

In [43]:
# Factory must return a new env each call — do not reuse a single env instance
env = DummyVecEnv([lambda: make_env()])

model = PPO("MlpPolicy", env, verbose=1, tensorboard_log=logs_path)

Using cpu device


In [44]:
# Train for 20,000 environment steps (adjust for longer/shorter runs)
model.learn(total_timesteps=20000)

Logging to Training/Logs/PPO_1
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 22       |
|    ep_rew_mean     | 22       |
| time/              |          |
|    fps             | 9373     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 28.4        |
|    ep_rew_mean          | 28.4        |
| time/                   |             |
|    fps                  | 5859        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.009043141 |
|    clip_fraction        | 0.116       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.686      |
|    explained_variance   | -0.000152   |

# 4. Save and Reload Model

Persist the trained weights to disk and load them back for inference.

In [45]:
# Path prefix for saved model files (.zip is appended automatically)
PPO_Model_Path = os.path.join(saved_path, "PPO_Model")

In [46]:
# Save policy, value network, and training metadata
model.save(PPO_Model_Path)

/Users/rohtash/MySpace/Services/AI/RL-Demo/venv/lib/python3.14/site-packages/stable_baselines3/common/save_util.py:284: UserWarning: Path 'Training/Saved Models' does not exist. Will create it.
  warnings.warn(f"Path '{path.parent}' does not exist. Will create it.")


In [47]:
# Free memory — useful before reloading to verify the save worked
del model

In [48]:
# Load saved model and attach it to the same vectorized environment
model = PPO.load(PPO_Model_Path, env=env)

# 5. Evaluation

Measure average reward over multiple episodes using SB3's built-in evaluator.

In [49]:
# render=False avoids SDL conflict between OpenCV (SB3) and pygame on macOS
# Returns (mean_reward, std_reward) over n_eval_episodes
evaluate_policy(model, env, n_eval_episodes=10, render=False)

(np.float64(459.5), np.float64(51.29961013497081))

In [50]:
env.close()

# 6. Test Model

Run one episode manually using the trained policy's predictions.

In [51]:
# Recreate vec env if closed in the previous cell
env = DummyVecEnv([lambda: make_env()])

obs = env.reset()
while True:
    action, _states = model.predict(obs)
    obs, rewards, done, info = env.step(action)
    # env.render() disabled — SDL conflict on macOS after SB3/OpenCV import
    if done:
        print("info", info)
        break

info [{'episode': {'r': 428.0, 'l': 428, 't': 0.05227}, 'TimeLimit.truncated': False, 'terminal_observation': array([-2.4216955 , -1.3923103 , -0.13034321, -0.34711474], dtype=float32)}]


In [52]:
env.close()

# 7. Viewing Logs in TensorBoard

Launch TensorBoard to inspect training curves (reward, loss, etc.).

In [53]:
# PPO_3 is the run folder created by SB3 (name increments each training run)
training_log_path = os.path.join(logs_path, "PPO_1")

In [54]:
# Open http://localhost:6006 in your browser after running this cell
!tensorboard --logdir={training_log_path}

TensorFlow installation not found - running with reduced feature set.
Serving TensorBoard on localhost; to expose to the network, use a proxy or pass --bind_all
TensorBoard 2.21.0 at http://localhost:6006/ (Press CTRL+C to quit)


OSError: [Errno 5] Input/output error

# 8. Adding a Callback to the Training Stage

Stop training early when reward crosses a threshold and save the best model.

In [55]:
from stable_baselines3.common.callbacks import EvalCallback, StopTrainingOnRewardThreshold

In [56]:
# Fresh Monitor-wrapped environment for callback-based training
env = DummyVecEnv([lambda: make_env()])

In [57]:
# Stop when mean eval reward >= 190 (near CartPole's max of ~500, but good for a demo)
stop_callback = StopTrainingOnRewardThreshold(reward_threshold=190, verbose=1)

eval_callback = EvalCallback(
    env,
    callback_on_new_best=stop_callback,
    eval_freq=10000,
    best_model_save_path=saved_path,
    verbose=1,
)

In [58]:
model = PPO('MlpPolicy', env, verbose=1, tensorboard_log=logs_path)

Using cpu device


In [59]:
model.learn(total_timesteps=20000, callback=eval_callback)

Logging to Training/Logs/PPO_2
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 19.8     |
|    ep_rew_mean     | 19.8     |
| time/              |          |
|    fps             | 9963     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 23.5        |
|    ep_rew_mean          | 23.5        |
| time/                   |             |
|    fps                  | 5906        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.007945505 |
|    clip_fraction        | 0.0781      |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.687      |
|    explained_variance   | 0.000428    |

In [61]:
# Load the best checkpoint saved by EvalCallback
model_path = os.path.join(saved_path, "best_model")
model = PPO.load(model_path, env=env)

In [62]:
# render=False — live render unsafe on macOS after SB3 import (SDL conflict)
evaluate_policy(model, env, n_eval_episodes=10, render=False)

(np.float64(327.3), np.float64(127.59706109468198))

In [63]:
env.close()

# 9. Changing Policies

Customize the neural network architecture via `policy_kwargs`.

In [64]:
# Larger shared architecture for policy (pi) and value (vf) networks
net_arch = [dict(pi=[128, 128, 128, 128], vf=[128, 128, 128, 128])]

In [65]:
# Reuse env from section 8 or recreate if needed
env = DummyVecEnv([lambda: make_env()])
model = PPO("MlpPolicy", env, verbose=1, policy_kwargs={"net_arch": net_arch})

Using cpu device


/Users/rohtash/MySpace/Services/AI/RL-Demo/venv/lib/python3.14/site-packages/stable_baselines3/common/policies.py:486: UserWarning: As shared layers in the mlp_extractor are removed since SB3 v1.8.0, you should now pass directly a dictionary and not a list (net_arch=dict(pi=..., vf=...) instead of net_arch=[dict(pi=..., vf=...)])
  warnings.warn(


In [66]:
model.learn(total_timesteps=20000, callback=eval_callback)

---------------------------------
| rollout/           |          |
|    ep_len_mean     | 21.5     |
|    ep_rew_mean     | 21.5     |
| time/              |          |
|    fps             | 8202     |
|    iterations      | 1        |
|    time_elapsed    | 0        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 28.9        |
|    ep_rew_mean          | 28.9        |
| time/                   |             |
|    fps                  | 4115        |
|    iterations           | 2           |
|    time_elapsed         | 0           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014686275 |
|    clip_fraction        | 0.192       |
|    clip_range           | 0.2         |
|    entropy_loss         | -0.682      |
|    explained_variance   | -0.00849    |
|    learning_rate        | 0.

# 10. Using an Alternate Algorithm

Swap PPO for DQN (Deep Q-Network) — an off-policy value-based method.

In [67]:
from stable_baselines3 import DQN

In [68]:
model = DQN('MlpPolicy', env, verbose=1, tensorboard_log=logs_path)

Using cpu device


In [69]:
model.learn(total_timesteps=20000, callback=eval_callback)

Logging to Training/Logs/DQN_1
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 22.2     |
|    ep_rew_mean      | 22.2     |
|    exploration_rate | 0.958    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 14606    |
|    time_elapsed     | 0        |
|    total_timesteps  | 89       |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 22       |
|    ep_rew_mean      | 22       |
|    exploration_rate | 0.916    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 5836     |
|    time_elapsed     | 0        |
|    total_timesteps  | 176      |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.516    |
|    n_updates        | 18       |
----------------------------------
----------------------------------
| rollout/            | 

In [70]:
dqn_path = os.path.join(saved_path, "DQN_Model")

In [71]:
model.save(dqn_path)

In [72]:
model = DQN.load(dqn_path, env=env)

In [73]:
# render=False — live render unsafe on macOS after SB3 import (SDL conflict)
evaluate_policy(model, env, n_eval_episodes=10, render=False)

(np.float64(9.2), np.float64(0.6))

In [74]:
env.close()